**A pendulum hanging from a cart on a spring** (Will's opening system for Chapter 6; Taylor's Figure 7.15).
A cart of mass $m$ rides on a frictionless track, tied to a wall by a spring of constant $k$; $x$ is the spring's extension.
A pendulum of length $L$ with a bob of mass $M$ hangs from the cart at angle $\phi$ from the vertical, so the bob is at

$$\mathbf{r}_{\rm bob} = (x + L\sin\phi,\; -L\cos\phi).$$

Two coordinates, two coupled motions. With Newton you would also need the string tension and the track's push on the cart, and you would have to eliminate them.
The Lagrangian (Chapter 7) gets the two equations of motion from $x$ and $\phi$ alone; the code below just solves them numerically.
Here $m = 1$ kg, $M = 0.5$ kg, $k = 10$ N/m, $L = 1$ m, and the cart starts from rest at $x = 0.7$ m with the bob at rest at angle $\phi_0$.
Drag $t$ to run the motion.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact

m, M = 1.0, 0.5           # cart, bob (kg)
k, L, g = 10.0, 1.0, 9.81
x0 = 0.7                  # cart starts here (m)

def rates(t, s):
    x, phi, v, w = s      # v = x-dot, w = phi-dot
    c, sn = np.cos(phi), np.sin(phi)
    # equations of motion: linear in x'' and phi''
    A = [[m + M, M * L * c],
         [c,     L]]
    b = [-k * x + M * L * w**2 * sn,
         -g * sn]
    xdd, phidd = np.linalg.solve(A, b)
    return [v, w, xdd, phidd]

def plot_cart(t=0.0, phi0_deg=0):
    start = [x0, np.radians(phi0_deg), 0, 0]
    sol = solve_ivp(rates, (0, 20), start,
                    dense_output=True, rtol=1e-8)
    x, phi = sol.sol(t)[:2]
    bx = x + L * np.sin(phi)
    by = -L * np.cos(phi)

    plt.figure(figsize=(10, 4))
    # wall and spring
    plt.plot([-3, -3], [-0.4, 0.4], 'k', lw=4)
    sx = np.linspace(-3, x - 0.5, 21)
    sy = 0.1 * (-1) ** np.arange(21)
    sy[0] = sy[-1] = 0
    plt.plot(sx, sy, 'b')
    plt.text(x - 1.5, 0.2, 'k')
    # cart
    plt.fill([x - 0.5, x + 0.5, x + 0.5, x - 0.5],
             [-0.25, -0.25, 0.25, 0.25], 'lightgray')
    plt.text(x, 0, 'm', ha='center', va='center')
    # pendulum
    plt.plot([x, bx], [0, by], 'k')
    plt.plot(bx, by, 'ro', ms=12)
    plt.text(bx + 0.1, by + 0.1, 'M')

    plt.title(fr'$t = {t:.2f}$ s,  $x = {x:.2f}$ m,  '
              fr'$\phi = {np.degrees(phi):.1f}^\circ$')
    plt.xlim(-3, 3); plt.ylim(-1.5, 0.5)
    plt.gca().set_aspect('equal')
    plt.xlabel('x (m)'); plt.ylabel('y (m)')
    plt.show()

interact(plot_cart, t=(0.0, 20.0, 0.05), phi0_deg=(-90, 90, 5));

## Interesting cases

Drag $t$ slowly from 0 to 20 s in each.

- $\phi_0 = 0$ (the default, Will's run): only the spring is stretched at the start. As the cart moves, the bob lags behind and starts to swing, and the swing pushes back on the cart. The motion never quite repeats: two coupled oscillations at once.
- $\phi_0 = 40^\circ$: cart and bob start displaced the same way. The pair moves nearly together, a single slow, nearly repeating oscillation (period about 2.8 s).
- $\phi_0 = -70^\circ$: the bob starts on the far side, leaning against the cart's displacement. Now cart and bob move in opposite directions, again nearly repeating but faster (period about 1.8 s).
- $\phi_0 = 90^\circ$: the bob starts level with the pivot. Large swings, and the cart is yanked around by the string.